### **Student Information**
GitHub ID:

Kaggle name:

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

***

# **Project Report**

**Syntax:** `#` creates the largest heading (H1).

---
**Syntax:** `---` creates a horizontal rule (a separator line).

## 1. Model Development (10 pts Required)

**Syntax:** `##` creates a secondary heading (H2).

**Describe briefly each section, you can add graphs/charts to support your explanations.**

### 1.1 Preprocessing Steps

**Syntax:** `###` creates a tertiary heading (H3).

[Content for Preprocessing]

**Example Syntax for Content:**
*   **Bold text:** `**text**`
*   *Italic text*: `*text*`
*   Bullet point list:
    * Item 1
    * Item 2

Markdown Syntax to Add Image: `![Description of the Image](./your_local_folder/name_of_the_image.png)`

![Example Markdown Syntax to Add Image](./pics/example_md_img.png)

### 1.2 Feature Engineering Steps

[Content for Feature Engineering]

### 1.3 Explanation of Your Model

[Content for Model Explanation]

### 1.4 AI Prompts, what and why you decide to use that technique that suggest by AI

[Content for AI Prompts and your reason]

---

**`From here on starts the code section for the competition.`**

---

# **Competition Code**

**how this section is organised**

- the default run takes a few seconds on cpu. it loads and checks the competition data, rebuilds the tf-idf baseline, and puts the final ensemble back together from the saved probabilities.
- we fine-tuned the transformers on kaggle with 2× T4 gpus. the training code is in section 3.2, but it's off here (`RUN_GPU_TRAINING = False`) since one 7B model takes about 2.5 gpu-hours per seed. section 3.3 loads the saved 5-fold out-of-fold and test probabilities.
- the competition files and saved probabilities are kept **outside this public repository** (`../kaggle_data/` and `../kaggle_work/runs/`). the competition data stays in those local folders.

## 1. Preprocessing Steps

In [1]:
# ---- 1.1 Join the post text, split assignments, and training labels ----
from pathlib import Path
import json, re
import numpy as np
import pandas as pd

DATA = Path("../kaggle_data")          # final_posts.json, data_identification.csv, emotion.csv (outside the repo)
RUNS = Path("../kaggle_work/runs")     # probabilities saved by the Kaggle GPU runs (outside the repo)
SUBS = Path("../kaggle_work/submissions")
LABELS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]   # fixed class order used everywhere
L2I = {l: i for i, l in enumerate(LABELS)}

posts = pd.DataFrame(json.load(open(DATA / "final_posts.json", encoding="utf-8")))   # post_id, text
ident = pd.read_csv(DATA / "data_identification.csv")                                  # post_id, set (train/test)
labels = pd.read_csv(DATA / "emotion.csv")                                             # post_id, label (train only)

df = posts.merge(ident, on="post_id")
train = df[df.set == "train"].merge(labels, on="post_id").reset_index(drop=True)
test = df[df.set == "test"].reset_index(drop=True)
y = train.label.map(L2I).values        # integer labels in LABELS order

print("posts", posts.shape, "| train", train.shape, "| test", test.shape)
print(train.label.value_counts().to_string())

posts (4800, 2) | train (3360, 4) | test (1440, 3)
label
joy         875
anger       674
sadness     628
disgust     454
surprise    441
fear        288


In [2]:
# ---- 1.2 Check the data as loaded ----
print("missing values :", int(df.isna().sum().sum()))
print("duplicate ids  :", int(df.post_id.duplicated().sum()))
print("empty texts    :", int((df.text.str.strip() == "").sum()))

# Group identical training posts and check whether their labels differ.
dup_groups = train.groupby("text").label.agg(["size", "nunique"])
dup_groups = dup_groups[dup_groups["size"] > 1]
print(f"train texts that appear more than once: {len(dup_groups)} groups, "
      f"{int(dup_groups['nunique'].gt(1).sum())} of them with conflicting labels")

# Check for exact matches across splits that could leak test examples into training.
overlap = set(train.text) & set(test.text)
print("texts shared by train and test:", len(overlap))

missing values : 0
duplicate ids  : 0
empty texts    : 0
train texts that appear more than once: 50 groups, 16 of them with conflicting labels
texts shared by train and test: 59


In [3]:
# ---- 1.3 Count text patterns and summarize post lengths ----
emoji_re = re.compile("[\U0001F300-\U0001FAFF☀-➿]")
checks = {
    "[NAME] placeholder (GoEmotions style)": df.text.str.contains(r"\[NAME\]", regex=True),
    "@<username> placeholder": df.text.str.contains("@<username>", regex=False),
    "##URL## placeholder": df.text.str.contains("##URL##", regex=False),
    "hashtag (#word)": df.text.str.contains(r"#\w", regex=True),
    "emoji": df.text.map(lambda t: bool(emoji_re.search(t))),
    "literal backslash-n": df.text.str.contains("\\n", regex=False),
    "curly quotes": df.text.str.contains("[‘’“”]", regex=True),
}
print(pd.DataFrame({"posts": {k: int(v.sum()) for k, v in checks.items()},
                    "share": {k: round(float(v.mean()), 3) for k, v in checks.items()}}).to_string())

n_words = df.text.str.split().str.len()
print("\nwords per post: median", int(n_words.median()), "| 95th pct", int(n_words.quantile(.95)),
      "| max", int(n_words.max()))

                                       posts  share
[NAME] placeholder (GoEmotions style)    347  0.072
@<username> placeholder                  136  0.028
##URL## placeholder                       31  0.006
hashtag (#word)                           72  0.015
emoji                                    159  0.033
literal backslash-n                        0  0.000
curly quotes                             322  0.067

words per post: median 13 | 95th pct 32 | max 510


In [4]:
# ---- 1.4 Preprocessing for the final models ----
# The TF-IDF baseline only lowercases text inside TfidfVectorizer.
# Words like "not", punctuation like "!", and emoji can all give clues about emotion.
# Each transformer gets the original text and uses its own tokenizer.
# We tested each option on Kaggle: normalizing curly quotes, replacing [NAME]
# with random names, dropping duplicates with conflicting labels, MLM augmentation,
# and pseudo-labeling. None raised 5-fold macro F1 by more than +0.005,
# so we left all of them out (see report 1.1).
def prepare_text(text: str) -> str:
    # Keep the original text, including hashtags, emoji, placeholders, and capitalization.
    return text

train["model_text"] = train.text.map(prepare_text)
test["model_text"] = test.text.map(prepare_text)
assert (train.model_text == train.text).all()

## 2. Feature Engineering Steps

In [5]:
# ---- 2.1 baseline features with word 1-2 grams and character 2-5 grams ----
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer

def tfidf_features(fit_texts, *other_texts):
    # word n-grams pick up phrases like "not happy"
    # char_wb n-grams help with typos, stretched words like "sooo", and hashtags stuck to words
    word = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, lowercase=True)
    char = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)
    X_fit = hstack([word.fit_transform(fit_texts), char.fit_transform(fit_texts)]).tocsr()
    others = [hstack([word.transform(t), char.transform(t)]).tocsr() for t in other_texts]
    return [X_fit] + others, (len(word.vocabulary_), len(char.vocabulary_))

(X_all,), sizes = tfidf_features(train.text)
print("TF-IDF features on all train rows: word", sizes[0], "+ char", sizes[1], "=", X_all.shape[1])

TF-IDF features on all train rows: word 7439 + char 27593 = 35032


In [6]:
# ---- 2.2 how we feed posts to the transformer models ----
# roberta and modernbert each use their own subword tokenizer, with a limit of 128 tokens
# for emollama-7b, we wrap each post in the prompt from the model card and ask for one of our 6 labels
# the classifier uses the hidden state of the last token (">>") to pick the emotion
# the prompt gives the model the task so that final position can capture the emotion
MAX_LEN_ENCODER = 128
MAX_LEN_DECODER = 256          # prompt (~45 tokens) + post (cut to 128 tokens)
EMO_PRE = ("Human: \nTask: Categorize the text's emotional tone as one of the given emotions "
           "(anger, disgust, fear, joy, sadness, surprise).\nText:")
EMO_SUF = "\nThis text contains emotions:\n\nAssistant:\n>>"

def emollama_input(text: str) -> str:
    # on kaggle we join the token ids directly, so we only tokenize the post once
    return EMO_PRE + " " + text + EMO_SUF

print(emollama_input(train.text[0]))

Human: 
Task: Categorize the text's emotional tone as one of the given emotions (anger, disgust, fear, joy, sadness, surprise).
Text: Thought it looked weird as well.
This text contains emotions:

Assistant:
>>


## 3. Model Implementation Steps

In [7]:
# ---- 3.1 try tf-idf with linear classifiers  on a 15% stratified hold-out ----
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, confusion_matrix

tr, va = train_test_split(train, test_size=0.15, stratify=train.label, random_state=42)
(X_tr, X_va), _ = tfidf_features(tr.text, va.text)
baselines = {
    "LinearSVC": LinearSVC(C=0.5, class_weight="balanced"),
    "LogReg": LogisticRegression(C=5, max_iter=3000, class_weight="balanced"),
}
for name, clf in baselines.items():
    pred = clf.fit(X_tr, tr.label).predict(X_va)
    print(f"{name:9s} hold-out macro F1 = {f1_score(va.label, pred, average='macro'):.4f}")
print(classification_report(va.label, baselines["LogReg"].predict(X_va), digits=3))

LinearSVC hold-out macro F1 = 0.5298


LogReg    hold-out macro F1 = 0.5465
              precision    recall  f1-score   support

       anger      0.520     0.515     0.517       101
     disgust      0.424     0.368     0.394        68
        fear      0.656     0.488     0.560        43
         joy      0.634     0.629     0.631       132
     sadness      0.549     0.660     0.599        94
    surprise      0.565     0.591     0.578        66

    accuracy                          0.560       504
   macro avg      0.558     0.542     0.547       504
weighted avg      0.560     0.560     0.557       504



In [8]:
# ---- 3.2 fine-tune the transformers on kaggle  (2x T4, turned off here) ----
# this is a shorter version of kaggle script.  every model use the same setup
# 5-fold stratified cv (split seed 42) gives oof probabilities for all 3360 train rows.
# each fold also predict the test set.  we use adamw (weight decay 0.01), 10% linear warm-up,
# gradient clipping at 1.0, fp16 autocast, batches of 16, and cross-entropy loss.
# we keep the last epoch.  picking the best one on the validation fold could make the oof score too high.
RUN_GPU_TRAINING = False

ENCODERS = [  # emotion models whose pretrained head already contains our 6 labels -> head rows are reused
    dict(name="r14_roberta_large_emo", model="j-hartmann/emotion-english-roberta-large"),
    dict(name="r26_roberta_base_goemo", model="SamLowe/roberta-base-go_emotions"),
    dict(name="r35_modernbert_base_goemo", model="cirimus/modernbert-base-go-emotions"),
    dict(name="r37_modernbert_large_goemo", model="fyaronskiy/ModernBERT-large-english-go-emotions"),
]
ENCODER_CFG = dict(lr=1e-5, epochs=4, max_len=MAX_LEN_ENCODER, bs=16)
DECODER = dict(name="r49_emollama7b_prompt", model="lzw1008/Emollama-7b",
               lr=1e-4, epochs=3, max_len=MAX_LEN_DECODER, bs=16,
               lora=dict(r=16, alpha=32, dropout=0.05, bits=4))   # QLoRA: 4-bit NF4 base + LoRA adapters
SEEDS = (42, 7, 123)   # every model is trained with 3 seeds and the seeds are averaged


def load_model(cfg, tok, device):
    import torch
    from transformers import AutoConfig, AutoModelForSequenceClassification
    mc = AutoConfig.from_pretrained(cfg["model"])
    if "lora" not in cfg:
        # copy the pretrained classifier rows for our 6 labels  in our label order
        m = AutoModelForSequenceClassification.from_pretrained(cfg["model"])
        name2id = {str(v).lower(): int(k) for k, v in mc.id2label.items()}
        idx = torch.tensor([name2id[l] for l in LABELS])
        parent, attr = (m.classifier, "out_proj") if hasattr(m.classifier, "out_proj") else (m, "classifier")
        old = getattr(parent, attr)
        new = torch.nn.Linear(old.in_features, len(LABELS))
        new.weight.data, new.bias.data = old.weight.data[idx].clone(), old.bias.data[idx].clone()
        setattr(parent, attr, new)
        return m.to(device)
    # use a 4-bit decoder with a new 6-way `score` layer  and lora on every attention and mlp projection
    from transformers import BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                           bnb_4bit_compute_dtype=torch.float16, llm_int8_skip_modules=["score"])
    m = AutoModelForSequenceClassification.from_pretrained(
        cfg["model"], num_labels=len(LABELS), quantization_config=q, dtype=torch.float16, device_map={"": 0})
    m.config.pad_token_id = tok.pad_token_id
    m = prepare_model_for_kbit_training(m, use_gradient_checkpointing=True)
    lo = cfg["lora"]
    return get_peft_model(m, LoraConfig(
        task_type="SEQ_CLS", r=lo["r"], lora_alpha=lo["alpha"], lora_dropout=lo["dropout"],
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        modules_to_save=["score"]))


def finetune_cv(cfg, seed, texts, labels_int, test_texts, n_folds=5, split_seed=42):
    # return oof probabilities [n_train, 6]  and test probabilities [n_test, 6] averaged across folds
    import random, torch
    from torch.utils.data import DataLoader
    from sklearn.model_selection import StratifiedKFold
    from transformers import AutoTokenizer, get_linear_schedule_with_warmup
    device = "cuda"
    tok = AutoTokenizer.from_pretrained(cfg["model"])
    if "lora" in cfg:
        tok.padding_side = "right"          # the classifier reads the last non-padding token
        texts = [emollama_input(t) for t in texts]; test_texts = [emollama_input(t) for t in test_texts]

    def batches(rows, labs=None, bs=16, shuffle=False):
        def collate(ix):
            enc = tok([rows[i] for i in ix], truncation=True, max_length=cfg["max_len"],
                      padding=True, return_tensors="pt")
            if labs is not None:
                enc["labels"] = torch.tensor([labs[i] for i in ix])
            return enc
        return DataLoader(list(range(len(rows))), batch_size=bs, shuffle=shuffle, collate_fn=collate)

    @torch.no_grad()
    def predict(m, dl):
        m.eval(); out = []
        for b in dl:
            with torch.autocast("cuda", dtype=torch.float16):
                out.append(torch.softmax(m(**{k: v.to(device) for k, v in b.items()}).logits.float(), -1).cpu().numpy())
        return np.concatenate(out)

    oof = np.zeros((len(texts), len(LABELS)), dtype=np.float32)
    test_prob = np.zeros((len(test_texts), len(LABELS)), dtype=np.float32)
    test_dl = batches(test_texts, bs=64)
    skf = StratifiedKFold(n_folds, shuffle=True, random_state=split_seed)
    for f, (tr_i, va_i) in enumerate(skf.split(texts, labels_int)):
        s = seed + f; random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
        m = load_model(cfg, tok, device)
        tr_dl = batches([texts[i] for i in tr_i], labels_int[tr_i].tolist(), cfg["bs"], shuffle=True)
        opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=cfg["lr"], weight_decay=0.01)
        steps = len(tr_dl) * cfg["epochs"]
        sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
        scaler = torch.amp.GradScaler("cuda")
        for ep in range(cfg["epochs"]):
            m.train()
            for b in tr_dl:
                b = {k: v.to(device) for k, v in b.items()}
                lab = b.pop("labels")
                with torch.autocast("cuda", dtype=torch.float16):
                    loss = torch.nn.functional.cross_entropy(m(**b).logits.float(), lab)
                opt.zero_grad(); scaler.scale(loss).backward()
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                scaler.step(opt); scaler.update(); sched.step()
        oof[va_i] = predict(m, batches([texts[i] for i in va_i], bs=64))          # last epoch
        test_prob += predict(m, test_dl) / n_folds
        print(f"{cfg['name']} seed {seed} fold {f}: macro F1 "
              f"{f1_score(labels_int[va_i], oof[va_i].argmax(1), average='macro'):.4f}")
        del m; torch.cuda.empty_cache()
    return oof, test_prob


if RUN_GPU_TRAINING:
    for c in ENCODERS:
        for s in SEEDS:
            finetune_cv({**c, **ENCODER_CFG}, s, train.model_text.tolist(), y, test.model_text.tolist())
    for s in SEEDS:
        finetune_cv(DECODER, s, train.model_text.tolist(), y, test.model_text.tolist())
else:
    print("GPU training skipped; loading the probabilities saved by the Kaggle runs below.")

GPU training skipped; loading the probabilities saved by the Kaggle runs below.


In [9]:
# ---- 3.3 load saved oof predictions  and score each model ----
def macro_f1(prob):
    return round(float(f1_score(y, prob.argmax(1), average="macro")), 4)

# the 5-fold oof files from kaggle has shape [epoch, row, class].
# for each model, take the last epoch from each of the 3 seeds  and average the probabilities.
MEMBER_OOF = {
    "r14 roberta-large emotion": ["run3/r14_rl_emo_full_base_full_s42", "run6/r14_rl_emo_full_base_full_s7",
                                  "run6/r14_rl_emo_full_base_full_s123"],
    "r26 roberta-base GoEmotions": ["run8/r26_goemo_full_full_s42", "run8/r26_goemo_full_full_s7",
                                    "run13/r26_goemo_full_full_s123"],
    "r35 ModernBERT-base GoEmotions": ["run15/r35_cirimus_mbb_full_full_s42", "run17/r35_cirimus_mbb_full_full_s7",
                                       "run17/r35_cirimus_mbb_full_full_s123"],
    "r37 ModernBERT-large GoEmotions": ["run15/r37_g3_fyaronskiy_mbl_full_full_s42",
                                        "run17/r37_g3_fyaronskiy_mbl_full_full_s7",
                                        "run17/r37_g3_fyaronskiy_mbl_full_full_s123"],
}
oof = {}
for name, files in MEMBER_OOF.items():
    seeds = [np.load(RUNS / f"{f}_oof_ep.npy")[-1] for f in files]     # last epoch of each seed
    oof[name] = np.mean(seeds, 0)                                       # average the 3 seeds
# emollama-7b's folds ran in separate kaggle sessions.  each seed file already combines their last epochs.
oof["r49 Emollama-7b QLoRA"] = np.mean(
    [np.load(RUNS / f"r49_emollama_prompt_s{s}_merged_oof_last.npy") for s in (42, 7, 123)], 0)

scores = pd.DataFrame({"5-fold OOF macro F1 (3 seeds averaged)": {k: macro_f1(v) for k, v in oof.items()}})
print(scores.to_string())

                                 5-fold OOF macro F1 (3 seeds averaged)
r14 roberta-large emotion                                        0.6279
r26 roberta-base GoEmotions                                      0.6251
r35 ModernBERT-base GoEmotions                                   0.6267
r37 ModernBERT-large GoEmotions                                  0.6138
r49 Emollama-7b QLoRA                                            0.6282


In [10]:
# ---- 3.4 average the models' class probabilities ----
# we add models one at a time  and only keep one if it raises oof macro f1 by >= 0.005.
# every model get the same weight. tuning weights on those same oof rows could overfit them.
encoders = [k for k in oof if not k.startswith("r49")]
ens_steps = [
    ("r14 alone", ["r14 roberta-large emotion"]),
    ("r14 + r26", encoders[:2]),
    ("S4 = 4 encoders", encoders),
    ("S4 + Emollama (final)", encoders + ["r49 Emollama-7b QLoRA"]),
]
for label, members in ens_steps:
    print(f"{label:24s} OOF macro F1 = {macro_f1(np.mean([oof[k] for k in members], 0))}")

final_oof = np.mean([oof[k] for k in encoders + ["r49 Emollama-7b QLoRA"]], 0)
assert np.allclose(np.mean([oof[k] for k in encoders], 0), np.load(RUNS / "S4_base_oof.npy"), atol=1e-6)
print()
print(classification_report(y, final_oof.argmax(1), target_names=LABELS, digits=3))

r14 alone                OOF macro F1 = 0.6279
r14 + r26                OOF macro F1 = 0.6428
S4 = 4 encoders          OOF macro F1 = 0.6537
S4 + Emollama (final)    OOF macro F1 = 0.6598

              precision    recall  f1-score   support

       anger      0.612     0.681     0.645       674
     disgust      0.593     0.443     0.507       454
        fear      0.758     0.632     0.689       288
         joy      0.741     0.826     0.781       875
     sadness      0.679     0.713     0.696       628
    surprise      0.678     0.608     0.641       441

    accuracy                          0.679      3360
   macro avg      0.677     0.651     0.660      3360
weighted avg      0.677     0.679     0.675      3360



In [11]:
# ---- 3.5 see which labels the final ensemble still mixes up on the oof rows ----
cm = pd.DataFrame(confusion_matrix(y, final_oof.argmax(1)),
                  index=[f"true {l}" for l in LABELS], columns=[f"pred {l}" for l in LABELS])
print(cm.to_string())

               pred anger  pred disgust  pred fear  pred joy  pred sadness  pred surprise
true anger            459            48         18        50            64             35
true disgust          123           201         14        43            46             27
true fear              26            20        182        20            33              7
true joy               45            17          8       723            39             43
true sadness           55            31         16        63           448             15
true surprise          42            22          2        77            30            268


In [12]:
# ---- 3.6 refit the same 5 models on all 3360 train rows for S13' ----
# for the test predictons we use the same setup, the same 3 seeds,  and the last epoch.
# these refits have no oof predictions, so before submitting we checked how often their
# test labels agreed with the 5-fold version and compared the class counts.
FULL = {
    "r14": "r14_rl_emo_full_base_full", "r26": "r26_goemo_full_full",
    "r35": "r35_cirimus_mbb_full_full", "r37": "r37_g3_fyaronskiy_mbl_full_full",
}
full_test = {k: np.mean([np.load(RUNS / f"run22/{v}_s{s}_all_test_all_ep.npy")[-1] for s in SEEDS], 0)
             for k, v in FULL.items()}
full_test["r49"] = np.mean([np.load(RUNS / f) [-1] for f in [
    "run27/r49_qlora_emollama7b_prompt_full_s42_all_test_all_ep.npy",
    "run27/r49_qlora_emollama7b_prompt_full_s7_all_test_all_ep.npy",
    "run35/r49_qlora_emollama7b_prompt_full_s123_all_test_all_ep.npy"]], 0)
final_test = np.mean(list(full_test.values()), 0)          # equal weight, 5 models

submission = pd.DataFrame({"id": test.post_id, "label": [LABELS[i] for i in final_test.argmax(1)]})
submitted = pd.read_csv(SUBS / "submission_S15_S9+r49allx3.csv")
print("identical to the file submitted to Kaggle:", submission.equals(submitted))
print(submission.label.value_counts(normalize=True).round(3).to_string())

identical to the file submitted to Kaggle: True
label
joy         0.278
anger       0.232
sadness     0.185
surprise    0.133
disgust     0.097
fear        0.075


In [13]:
# ---- 3.7 leaderboard scores for the main submissions ----
# oof is the 5-fold macro f1 from the cells above or the experiment log.
# public and private scores comes from kaggle's "My Submissions" page after the competition closed.
# the 5-fold S4 + Emollama submission used 1 Emollama seed (0.6585).
# cell 19 averages 3 Emollama seeds,  which gives 0.6598.
results = pd.DataFrame([
    ("TF-IDF + LogReg baseline",           0.547,  0.5045, 0.4908, ""),
    ("r1 distilroberta-emotion",           0.6206, 0.6163, 0.6100, ""),
    ("S4 (4 encoders x 3 seeds)",          0.6537, 0.6556, 0.6541, ""),
    ("S4 + Emollama (5-fold models)",      0.6585, 0.6717, 0.6663, ""),
    ("S9 (S4 refit on all train)",         None,   0.6699, 0.6602, "final"),
    ("S13' (S9 + Emollama x3, all train)", None,   0.6722, 0.6629, "final"),
], columns=["submission", "OOF macro F1", "public F1", "private F1", "selected"])
print(results.to_string(index=False))

                        submission  OOF macro F1  public F1  private F1 selected
          TF-IDF + LogReg baseline        0.5470     0.5045      0.4908         
          r1 distilroberta-emotion        0.6206     0.6163      0.6100         
         S4 (4 encoders x 3 seeds)        0.6537     0.6556      0.6541         
     S4 + Emollama (5-fold models)        0.6585     0.6717      0.6663         
        S9 (S4 refit on all train)           NaN     0.6699      0.6602    final
S13' (S9 + Emollama x3, all train)           NaN     0.6722      0.6629    final
